In [1]:
from pathlib import Path
import sys

%load_ext autoreload
%autoreload 2

cwd = Path.cwd().resolve()

# Locate helper_function by searching upward first, then common workspace locations.
def find_helper_folder(start: Path) -> Path:
    # 1) Directly in current/parent tree
    for p in [start, *start.parents]:
        candidate = p / "helper_function"
        if candidate.is_dir():
            return candidate

    raise FileNotFoundError("Could not find helper_function folder")


helper_path = find_helper_folder(cwd)
project_root = helper_path.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(helper_path) not in sys.path:
    sys.path.insert(0, str(helper_path))

print("Project root:", project_root)
print("Current working directory:", cwd)
print("helper_function path:", helper_path)
print("helper_function exists:", helper_path.exists())

Project root: /home/zx296/Desktop/trap_sim_nullspace
Current working directory: /home/zx296/Desktop/trap_sim_nullspace/lionix_gate/run_file/rf
helper_function path: /home/zx296/Desktop/trap_sim_nullspace/helper_function
helper_function exists: True


In [ ]:
# Step 3: Generate grid file for ion height scan
import numpy as np
from grid_file_generation import generate_grid_file

# use a odd number of points to ensure the center point is included

grid_file = generate_grid_file(
    x_scan=[0],
    y_scan=np.linspace(-5, 5, 101),
    z_scan=np.linspace(45, 55, 101),
    filename="yz_scan.h5",
    gridID=0
)
grid_file

z_scan.h5


'z_scan.h5'

In [11]:
# Step 4: Find ion height
from input_deck_generation import generate_input_deck

electrode_num = 27
rf_electrode_index = [13]
gnd_electrode_index = [2, 14, 26]
#excitation_electrodes = range(1, electrode_num + 1) - set(gnd_electrode_index) - set(rf_electrode_index)
excitation_electrodes = rf_electrode_index

input_deck = generate_input_deck(mesh_file="chip.inp",
    input_deck_filename="rf_sim.in",
    grid_file="z_scan.h5",
    excitation_electrodes=excitation_electrodes
)
input_deck

rf_sim.in


'rf_sim.in'

In [12]:
# Step 5: Post-process result file into clean output HDF5
from pathlib import Path
from post_processing import post_process

output_dir = project_root / "lionix_gate" / "output_file" / "rf"
output_dir.mkdir(parents=True, exist_ok=True)

output_h5 = post_process(
    result_filename="rf_sim.in.h5",
    output_filename=str(output_dir / "rf_sim_out.h5"),
    gridID=0,
    verbose=True,
)
output_h5

Getting electrodes from rf_sim.in.h5
Getting grid from rf_sim.in.h5
Reading gridId 0 from z_scan.h5
1
[-5.0e-06 -4.5e-06 -4.0e-06 -3.5e-06 -3.0e-06 -2.5e-06 -2.0e-06 -1.5e-06
 -1.0e-06 -5.0e-07  0.0e+00  5.0e-07  1.0e-06  1.5e-06  2.0e-06  2.5e-06
  3.0e-06  3.5e-06  4.0e-06  4.5e-06  5.0e-06]
[-5.0e-06 -4.5e-06 -4.0e-06 -3.5e-06 -3.0e-06 -2.5e-06 -2.0e-06 -1.5e-06
 -1.0e-06 -5.0e-07  0.0e+00  5.0e-07  1.0e-06  1.5e-06  2.0e-06  2.5e-06
  3.0e-06  3.5e-06  4.0e-06  4.5e-06  5.0e-06]
[4.50e-05 4.55e-05 4.60e-05 4.65e-05 4.70e-05 4.75e-05 4.80e-05 4.85e-05
 4.90e-05 4.95e-05 5.00e-05 5.05e-05 5.10e-05 5.15e-05 5.20e-05 5.25e-05
 5.30e-05 5.35e-05 5.40e-05 5.45e-05 5.50e-05]
Getting potentials for grid 0, electrode 13 from rf_sim.in.h5
Getting gridType from rf_sim.in.h5
Reading grid type for gridId 0 from z_scan.h5
Getting grid from rf_sim.in.h5
Reading gridId 0 from z_scan.h5
/home/zx296/Desktop/trap_sim_nullspace/lionix_gate/output_file/rf/rf_sim_out.h5


'/home/zx296/Desktop/trap_sim_nullspace/lionix_gate/output_file/rf/rf_sim_out.h5'